# 통신데이터 통합

역할: 분석에 사용할 parquet 후보 파일을 테이블별 1개로 후보로 정리한다.

이 노트북에는 파일 생성/확정에 필요한 코드만 둔다. 행 수, 기간, 결측, 코드값 검증은 `통신데이터_검증.ipynb`에서 수행한다.

## 전체 t 데이터 통합 범위

기존 작업에서는 T4~T27 범위의 통신 데이터를 확인했다.

- T4~T23: 구조 파악, 보조 지표, EDA 검토 대상
- T13/T24/T25/T26/T27: 데이터가 커서 따로 보고 확인해야될 파일
- T20/T21: 현재 CSV 통합 파일로 남아 있어 parquet 분석 후보 파일에서는 제외

In [ ]:
import pandas as pd

ALL_T_FILES = {
    'T4':  {'file': 't4_2023_2025_all_date_final.parquet',  'format': 'parquet', 'unit': '시군구', 'meaning': '도착지 + 목적', 'decision': '구조 파악 / 보조'},
    'T5':  {'file': 't5_2023_2025_all_date_final.parquet',  'format': 'parquet', 'unit': '행정동', 'meaning': '도착지 + 목적', 'decision': '구조 파악 / 보조'},
    'T6':  {'file': 't6_2023_2025_all_date_final.parquet',  'format': 'parquet', 'unit': '시군구', 'meaning': '도착지 + 교통수단', 'decision': '구조 파악 / 보조'},
    'T7':  {'file': 't7_2023_2025_all_date_final.parquet',  'format': 'parquet', 'unit': '행정동', 'meaning': '도착지 + 교통수단', 'decision': '구조 파악 / 보조'},
    'T8':  {'file': 't8_2023_2025_all_date_final.parquet',  'format': 'parquet', 'unit': '시군구', 'meaning': '출발지 + 목적', 'decision': '구조 파악 / 보조'},
    'T9':  {'file': 't9_2023_2025_all_date_final.parquet',  'format': 'parquet', 'unit': '행정동', 'meaning': '출발지 + 목적', 'decision': '구조 파악 / 보조'},
    'T10': {'file': 't10_2023_2025_all_date_final.parquet', 'format': 'parquet', 'unit': '시군구', 'meaning': '출발지 + 교통수단', 'decision': '구조 파악 / 보조'},
    'T11': {'file': 't11_2023_2025_all_date_final.parquet', 'format': 'parquet', 'unit': '행정동', 'meaning': '출발지 + 교통수단', 'decision': '구조 파악 / 보조'},
    'T12': {'file': 't12_2023_2025_all_final_v2.parquet',   'format': 'parquet', 'unit': '시군구 OD', 'meaning': '출발지->도착지 + 목적', 'decision': '구조 파악 / 보조'},
    'T13': {'file': 't13_seongnam_final.parquet',           'format': 'parquet', 'unit': '행정동 OD', 'meaning': '출발지->도착지 + 목적', 'decision': '예측 변수 후보'},
    'T14': {'file': 't14_2023_2025_all_final_v2.parquet',   'format': 'parquet', 'unit': '시군구 OD', 'meaning': '출발지->도착지 + 교통수단', 'decision': '구조 파악 / 보조'},
    'T16': {'file': 't16_2023_2025_all_date_final.parquet', 'format': 'parquet', 'unit': '시군구', 'meaning': '도착지 + 목적 + 체류시간', 'decision': '구조 파악 / 보조'},
    'T20': {'file': 't20_2023_2025_all_date_final.csv',     'format': 'csv', 'unit': '기타', 'meaning': '통합 CSV 참고 파일', 'decision': '참고용 / 분석 후보 제외'},
    'T21': {'file': 't21_2023_2025_all_date_final.csv',     'format': 'csv', 'unit': '기타', 'meaning': '통합 CSV 참고 파일', 'decision': '참고용 / 분석 후보 제외'},
    'T22': {'file': 't22_2023_2025_all_date_final.parquet', 'format': 'parquet', 'unit': '행정동', 'meaning': '시간대 + 성별/연령 + 내외국인', 'decision': '구조 파악 / 보조'},
    'T23': {'file': 't23_2023_2025_all_date_final.parquet', 'format': 'parquet', 'unit': '시군구', 'meaning': '시간대 + 목적 유동인구', 'decision': '구조 파악 / 보조'},
    'T24': {'file': 't24_seongnam_final.parquet', 'format': 'parquet', 'unit': '행정동', 'meaning': '시간대 + 목적 유동인구', 'decision': '예측 변수 후보'},
    'T25': {'file': 't25_seongnam_final.parquet',           'format': 'parquet', 'unit': '시군구 OD', 'meaning': '출발지->도착지 + 시간대 + 목적 + 교통수단', 'decision': '예측 변수 후보'},
    'T26': {'file': 't26_seongnam_final.parquet',           'format': 'parquet', 'unit': '행정동', 'meaning': '도착지 + 시간대 + 목적 + 교통수단 + 체류시간', 'decision': '예측 변수 후보'},
    'T27': {'file': 't27_seongnam_final.parquet',           'format': 'parquet', 'unit': '행정동', 'meaning': '출발지 + 시간대 + 목적 + 교통수단 + 체류시간', 'decision': '예측 변수 후보'},
}

all_t_inventory = pd.DataFrame([
    {'table': table, **info}
    for table, info in ALL_T_FILES.items()
])
all_t_inventory

In [ ]:
from pathlib import Path
import os
import shutil
import pandas as pd

DATA_DIR = Path('../data')
FINAL_MAP = {
    'T13': {
        'source': DATA_DIR / 't13_2023_2025_all_final_v2.parquet',
        'final': DATA_DIR / 't13_seongnam_final.parquet',
        'purpose': '행정동/시군구 기준 이동량 구조',
    },
    'T24': {
        'source': DATA_DIR / 't24_2023_2025_all_date_final.parquet',
        'final': DATA_DIR / 't24_seongnam_final.parquet',
        'purpose': '유동인구/경제활동 연령층 proxy',
    },
    'T25': {
        'source': DATA_DIR / 't25_2023_2025_all_final_v2.parquet',
        'final': DATA_DIR / 't25_seongnam_final.parquet',
        'purpose': '유입/유출 패턴 중심',
    },
    'T26': {
        'source': DATA_DIR / 't26_2023_2025_all_final.parquet',
        'final': DATA_DIR / 't26_seongnam_final.parquet',
        'purpose': '체류시간 중심',
    },
    'T27': {
        'source': DATA_DIR / 't27_2023_2025_all_final.parquet',
        'final': DATA_DIR / 't27_seongnam_final.parquet',
        'purpose': '이동수단 + 목적 + 체류 특성',
    },
}

In [ ]:
for table, info in FINAL_MAP.items():
    source = info['source']
    final = info['final']
    if not source.exists():
        raise FileNotFoundError(f'{table} source not found: {source}')
    if final.exists():
        print(f'{table}: already exists -> {final.name}')
        continue
    try:
        os.link(source, final)
        print(f'{table}: hardlink created -> {final.name}')
    except OSError:
        shutil.copy2(source, final)
        print(f'{table}: copied -> {final.name}')

In [ ]:
final_files = pd.DataFrame([
    {
        '테이블명': table,
        '후보 파일명': info['final'].name,
        '원본 후보 파일': info['source'].name,
        '사용 후보 목적': info['purpose'],
    }
    for table, info in FINAL_MAP.items()
])
final_files

## 통합 판단 기준

통합 노트북에서는 모든 분석을 수행하지 않고, 파일을 어떤 이름으로 후보로 정해서 이후 검증/EDA에서 쓸지 정한다.  
기존 작업에서 T4~T27 전체를 확인했지만, 모든 테이블을 같은 수준으로 분석 후보에 쓰는 것은 아니므로 아래처럼 구분한다.

| 구분 | 테이블 | 처리 방향 |
|---|---|---|
| 구조 파악 / 보조 | T4~T12, T14, T16, T22, T23 | 전체 통신 데이터 구조 이해와 비교용으로 유지 |
| 예측 변수 후보 | T13, T24, T25, T26, T27 | EDA에서 예측용 변수 후보로 직접 연결 |
| 참고 CSV | T20, T21 | parquet 분석 후보 파일이 아니라 참고용 CSV로 유지 |

이 기준 때문에 파일명 후보 정리은 `*_seongnam_final.parquet` 형태로 별도 표시하고, 검증/EDA에서는 이 후보명을 우선 사용한다.

In [ ]:
# 예측 변수 후보 파일만 별도 후보명으로 관리한다.
FINAL_VARIABLE_FILES = {
    'T13': 't13_seongnam_final.parquet',
    'T24': 't24_seongnam_final.parquet',
    'T25': 't25_seongnam_final.parquet',
    'T26': 't26_seongnam_final.parquet',
    'T27': 't27_seongnam_final.parquet',
}

final_variable_files = pd.DataFrame([
    {'table': table, 'final_file': file_name}
    for table, file_name in FINAL_VARIABLE_FILES.items()
])
final_variable_files

## 통합 산출물 관리 원칙

- 같은 테이블의 중복 버전이 여러 개 있으면 분석에 쓸 파일명을 하나만 후보로 정리한다.
- 파일명은 `t번호_seongnam_final.parquet` 형태로 맞춘다.
- 대용량 parquet는 불필요하게 복사하지 않고 하드링크를 우선 사용한다.
- 원본 후보 파일과 후보 파일의 행 수 일치는 검증 노트북에서 확인한다.
- 기존 작업 기록은 통합 노트북에 다시 섞지 않고 `통신데이터_작업기록.ipynb`에 보관한다.

## 모델링 연결 기준

통합 노트북에서는 아직 모델 feature를 만들지 않는다. 대신 EDA 결과를 기준으로 모델링 후보 테이블을 아래처럼 구분해 둔다.

| 구분 | 테이블 | 이후 처리 |
|---|---|---|
| 기본 feature 후보 | T24 | 행정동-분기 마스터 테이블의 기본 유동인구 변수 후보 |
| 핵심 추가 후보 | T13, T26 | 외부유입/체류시간 변수 생성 후보 |
| 선택 추가 후보 | T27 | 이동수단/접근성 변수 후보 |
| 보조 후보 | T25 | 시군구 단위 유입 압력 참고 |

팀 마스터 키가 확정되면 `base_quarter + admi_cd` 기준으로 feature table 후보를 생성한다.